# Практика: как измерять качество движения видео с помощью VMBench

Мы будем запускать пять разных evaluator’ов и смотреть, какие свойства видео они считают хорошими или плохими.
Сначала сформулируйте гипотезу, затем сравните числа со своим визуальным впечатлением.

```text
video
 ├── PAS → amplitude
 ├── OIS → object integrity
 ├── TCS → temporal coherence
 ├── MSS → smoothness
 └── CAS → commonsense
```

Источники: [официальный код](https://github.com/AMAP-ML/VMBench),
[статья ICCV 2025](https://openaccess.thecvf.com/content/ICCV2025/html/Ling_VMBench_A_Benchmark_for_Perception-Aligned_Video_Motion_Generation_ICCV_2025_paper.html),
[полный текст arXiv](https://arxiv.org/html/2503.10076v2).
Проверенная ревизия: `9772be365784bc3f641d64fca018b1debb749116`.

**Colab:** выберите Runtime → Change runtime type → T4 GPU, затем Runtime → Run all.
Сначала скачивается repo вместе с VMBench submodule; затем без demo URL появится диалог загрузки видео. Drive не нужен. Начните с одного короткого ролика
(примерно 3–6 секунд, 40–60 кадров, 320×240 или 512×384); длинные HD-видео гораздо тяжелее.
Вход автоматически не уменьшается и не обрезается.

**Граница проверки:** notebook проверен на CPU: imports, структура, адаптеры и учебные операции.
Полный inference с checkpoint’ами на Colab T4 автором notebook не проверен.
MSS в исходном FP16 требует больше VRAM, чем есть на T4. По умолчанию он читает результат преподавателя,
если такой CSV предоставлен; иначе показывает N/A. Есть явно экспериментальный 8-bit режим.
Не путайте успешно выполненную ячейку с успешно рассчитанной метрикой: проверяйте `status` и `source`.


## Загружаем workshop с submodule

Первая ячейка скачивает весь repo по HTTPS с `--recurse-submodules`, включая `utils.py` и VMBench.
Оба repo должны быть публичными. SSH-ключ, Drive и отдельный upload модуля не нужны.
При локальном запуске используется текущий checkout; загрузка видео будет отдельным шагом.


In [ ]:
from pathlib import Path
import importlib.util
import hashlib
import os
import subprocess
import sys

WORKSHOP_URL = 'https://github.com/KK-one/vmbench-workshop.git'
WORKSHOP_REF = '4ff6b33a125f794692bce805d6318d6a660d38a4'  # Проверенная версия repo и submodule.
WORKSHOP_DIR = Path('/content/workshop')
local_project = Path.cwd()
git_env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
if (local_project / 'utils.py').is_file() and (local_project / '.gitmodules').is_file():
    WORKSHOP_DIR = local_project  # Do not checkout/change a local working copy.
else:
    if not (WORKSHOP_DIR / '.git').exists():
        result = subprocess.run(['git', 'clone', '--recurse-submodules', WORKSHOP_URL,
                                 str(WORKSHOP_DIR)], capture_output=True, text=True, env=git_env)
        if result.returncode:
            raise RuntimeError('Не удалось скачать workshop. Сделайте repo Public на GitHub и повторите ячейку. '
                               + result.stderr[-500:])
    subprocess.run(['git', '-C', str(WORKSHOP_DIR), 'checkout', '--detach', WORKSHOP_REF], check=True, env=git_env)
subprocess.run(['git', '-C', str(WORKSHOP_DIR), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
utils_path = WORKSHOP_DIR / 'utils.py'
if not utils_path.is_file():
    raise RuntimeError('В checkout отсутствует utils.py; проверьте версию workshop.')
module_hash = hashlib.sha256(utils_path.read_bytes()).hexdigest()
loaded = sys.modules.get('utils')
if (loaded is not None and Path(getattr(loaded, '__file__', '')).resolve() == utils_path.resolve()
        and getattr(loaded, '_workshop_module_hash', None) == module_hash):
    utils = loaded
else:
    spec = importlib.util.spec_from_file_location('utils', utils_path)
    utils = importlib.util.module_from_spec(spec)
    sys.modules['utils'] = utils
    spec.loader.exec_module(utils)
    utils._workshop_module_hash = module_hash
from utils import (WorkshopConfig, configure, compute_pas, compute_ois, compute_tcs,
                   compute_mss, compute_cas, get_video_info, show_video, clear_gpu,
                   compare_variants, evaluate_video, validate_submission)
print('Workshop:', WORKSHOP_DIR, '| VMBench:', WORKSHOP_DIR / 'vendor/vmbench')


## 1. Проверяем GPU

Если GPU отсутствует, включите его в настройках Colab. Просмотр видео, таблицы и задания доступны и на CPU;
evaluator’ы пропускаются с понятным сообщением. Эта ячейка использует PyTorch самого Colab.


In [ ]:
import subprocess, sys, os, json, time, gc, hashlib, shutil, traceback
from pathlib import Path
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi отсутствует: GPU runtime не выбран.')
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    print('CUDA доступна:', GPU_AVAILABLE)
    if GPU_AVAILABLE:
        print('GPU:', torch.cuda.get_device_name(0))
        GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
        print(f'VRAM: {GPU_GIB:.2f} GiB; BF16: {torch.cuda.is_bf16_supported()}')
    else:
        GPU_GIB = 0.0
        print('Для расчёта выберите Runtime → Change runtime type → T4 GPU.')
except ImportError:
    GPU_AVAILABLE, GPU_GIB = False, 0.0
    print('PyTorch не найден. Откройте notebook в Google Colab с GPU.')
print('Python notebook:', sys.version.split()[0])


## 2. Одна конфигурация для занятия

Для своих роликов замените `VIDEOS` или используйте upload. Значение URL может быть пустым.
`subject_noun` — **английское название видимого субъекта**, используемое GroundingDINO.
OIS в опубликованном CLI предназначен для человека: для животного или предмета укажите соответствующий `subject_type`.
Не используйте название файла как описание сцены. Для новых upload по умолчанию выбран `person / human`;
после загрузки исправьте `VIDEO_META` и повторите ячейки конфигурации результатов и нужной метрики.

`METRICS_TO_RUN` управляет отдельными учебными запусками и загрузкой весов.
`config.mss_mode='precomputed'` — безопасный исходный вариант для T4.
`config.mss_mode='fp16'` — исходная точность на GPU с ≥22 GiB; `config.mss_mode='8bit_experimental'` — официальный quantized loader,
числа могут отличаться и не считаются точным воспроизведением benchmark.


In [ ]:
VIDEOS = {'real_normal': '', 'generated': '', 'fast': '', 'slow': ''}
VIDEO_META = {
    # 'generated.mp4': {'subject_noun': 'dog', 'subject_type': 'animal',
    #                   'prompt': 'A dog runs on grass.'},
}
METRICS_TO_RUN = ['PAS', 'OIS', 'TCS', 'MSS', 'CAS']

config = WorkshopConfig(
    repo_dir=WORKSHOP_DIR / 'vendor/vmbench',
    videos=VIDEOS, video_meta=VIDEO_META, metrics_to_run=METRICS_TO_RUN,
    input_mode='auto',                   # auto / upload / demo
    mss_mode='precomputed',              # precomputed / fp16 / 8bit_experimental
    precomputed_url='',                 # CSV преподавателя, если есть
    create_variants=False,
    run_variant_comparisons=False,
    run_dataset=True,
    download_csv=False,
    gpu_available=GPU_AVAILABLE, gpu_gib=GPU_GIB,
)
workshop = configure(config)


### 2.1. Проверяем VMBench submodule

VMBench уже загружен в `vendor/vmbench` первой ячейкой. Здесь проверяем закреплённый commit
и необходимые файлы; отдельной копии VMBench не создаём.
Ошибки setup сохраняются в `/content/vmbench_runs/`.


In [ ]:
workshop.install_repository()


### 2.2. Системные зависимости через apt

FFmpeg нужен для декодирования и временного resampling. Компилятор нужен только для маленького
опционального CUDA extension SAM2; MMCV скачивается готовым wheel.


In [ ]:
workshop.install_system_dependencies()


### 2.3. pip: вспомогательные пакеты и изоляция

PyTorch notebook не обновляем. У evaluator’ов отдельные venv с Python 3.11, устанавливаемым через uv
(без conda). Это сознательное отступление от предпочтения использовать Python Colab:
старый Transformers/Q-Align и готовый ABI MMCV надежнее изолировать от меняющегося Colab.
Для PAS/TCS/CAS/MSS — PyTorch 2.5.1 + CUDA 12.4; для OIS — PyTorch 2.1.0 + CUDA 12.1,
совместимый с проверенным wheel MMCV 2.1.0. CUDA runtime есть в torch wheels, отдельный toolkit
не нужен для основных операций; `nvcc` требуется только для опционального SAM2 extension.
Две среды занимают несколько дополнительных GB диска. OIS среда устанавливается при первом OIS.


In [ ]:
workshop.install_host_packages()


### 2.4. Дополнительные пакеты evaluator’ов и проверка imports

Не запускаем все upstream requirements вслепую: VideoMAEv2 требует старые training-пакеты,
Q-Align также устанавливает UI и несовместимые зависимости. Ни training, ни web UI для оценки не нужны.
Используем vendored исходники и минимальные inference dependencies. GroundingDINO использует свой
официальный PyTorch fallback deformable attention: нет сборки старого CUDA extension.
Он выполняет ту же операцию, но возможны небольшие численные различия с fused CUDA kernel.

После успешной установки создаётся marker; повторный Run all не переустанавливает среду.
Первый setup обычно занимает **10–25 минут без checkpoint’ов** — оценка, не замер T4.


In [ ]:
workshop.install_evaluators()


## 3. Checkpoint’ы: скачиваем только при вызове метрики

Функции ниже скачивают веса в `/content/workshop/vendor/vmbench/.cache`, проверяют размер и переиспользуют полный файл.
`.part` при неудаче удаляется, окончательное имя появляется только после завершения загрузки.
Для Hugging Face фиксированы revision и размеры из API. Загрузка не требует токена для этих публичных моделей.

| Checkpoint | Метрики | Размер файла, GB (10⁹ bytes) |
|---|---|---:|
| GroundingDINO Swin-B | PAS, TCS | 0.938 |
| BERT base, safetensors + tokenizer/config | PAS, TCS | 0.441 |
| SAM ViT-H | PAS | 2.565 |
| CoTracker3 scaled_offline | PAS, TCS | 0.102 |
| SAM2.1 Hiera large | TCS | 0.898 |
| RTMDet person + RTMPose body7 | OIS | 0.099 + 0.055 |
| One-Align: два shards | MSS | 16.409 |
| vit_g_vmbench.pt (включает training state) | CAS | 14.163 |

Размер CAS файла **не равен VRAM**: модель содержит 1 011 617 797 параметров (~3.77 GiB FP32).
Для CAS используем `torch.load(mmap=True)` и читаем только `module`; optimizer на GPU не переносится.
Исходные One-Align shards содержат BF16 tensors; загрузчик VMBench приводит их к FP16 — число bytes
не уменьшается. Только веса MSS занимают около **15.28 GiB**, больше памяти T4.


In [ ]:
print('Веса скачиваются при вызове выбранной метрики; настройки находятся в utils.py.')


## 4. Видео: upload или demo URL

Поддерживаются MP4, MOV, WEBM. MOV/WEBM преобразуются в MP4/H.264 для SAM2; это перекодирование,
не изменение скорости. MP4 переиспользуется как есть. Избегайте HDR/необычных codec; FFmpeg-ошибка
останется в логе. Несколько файлов можно загрузить за один раз. Пустая загрузка не ломает Run all.

Demo URL должны отдавать **сам файл**, не страницу YouTube/Google Drive. Преподаватель заменяет их в `VIDEOS`.


In [ ]:
input_paths = workshop.load_videos()
# Для дополнительных файлов: workshop.upload_more()


## 5. Просмотр и метаданные

FPS и длительность помогут интерпретировать speed experiment. Для VFR значение FPS усреднённое.
Полный просмотр длинных файлов через `embed=True` увеличивает размер HTML вывода; для занятия берите короткие ролики.


In [ ]:
video_info_df = workshop.preview_videos()


## 6. Optional: FAST ×2, SLOW ×0.5, frame drop и frame duplication

Установите `config.create_variants=True` в конфигурации, если нужен этот эксперимент.
Все варианты перекодируются из одного исходника одинаковым H.264/CRF, звук удаляется.
FAST: `setpts=0.5*(PTS-STARTPTS)` и затем `fps=исходный FPS` — длительность уменьшается,
часть кадров отбрасывается. SLOW: `setpts=2*(PTS-STARTPTS)` с тем же FPS — длительность растёт,
кадры дублируются, оптический flow не интерполируется.

DROP: удаляем каждый второй кадр и воспроизводим оставшиеся с половинным FPS — длительность примерно прежняя.
DUP: этот поток возвращаем к исходному FPS дублированием кадров. DROP и DUP имеют близкое зрительное движение,
но разное число декодированных кадров. Это **новые экспериментальные входы**, не изменение определения evaluator’а.
Подумайте, измеряет ли метрика секунды, число кадров или их перемещения, прежде чем ожидать ranking.


In [ ]:
VARIANTS = workshop.prepare_variants()


## 7. Выбираем входы и готовим отдельные процессы

`VIDEO_PATH` — один ролик для первого вызова каждой метрики. `VIDEO_PATHS` — список для итоговой таблицы.
При необходимости замените эти две строки. Субъект задаётся в `VIDEO_META` основной конфигурации.
Существующий `create_meta_info.py` связывает только номера официальных 1050 prompts; для произвольных
upload создаём совместимый JSON (`filepath`, `subject_noun`, `prompt`). Метрики остаются официальными.


In [ ]:
VIDEO_PATH, VIDEO_PATHS = workshop.select_videos()
# При необходимости выберите другой VIDEO_PATH из VIDEO_PATHS.


### Исправления интеграции upstream

Это адаптер проверенной ревизии, не новое API VMBench. Оригинальные файлы не перезаписываются.
Адаптер в `utils.py` применяет патчи к копии текста в дочернем процессе; проверяем, что ожидаемая строка присутствует ровно один раз.

1. **PAS:** `apply_boxes_torch` получает `(C,H)` из normalized tensor вместо `(H,W)` исходного изображения.
   Исправляем размер на `image_array.shape[:2]`. Это исправление масок **может изменить score** относительно buggy CLI.
   Сумма смещений, нормализация диагональю и правило компенсации камеры сохранены. PAS здесь сырой,
   не ограничен единицей; обещать шкалу 0–100 по описанию paper нельзя.
2. **GroundingDINO:** запускаем уже включённую в проект PyTorch реализацию deformable attention на CUDA,
   когда `_C` не собран. Алгоритм не заменяем; fused kernel и fallback не обязаны быть побитово равны.
3. **OIS:** без person bbox MMPose может перейти к whole-image pose; без instances `pose_utils` обращается
   к `instances[0]`. Вместо такого результата возвращаем N/A, если хотя бы один кадр не содержит валидную
   human detection. Это политика применимости, не заполнение отсутствующих кадров выдуманными позами.
4. **TCS:** выключаем безусловный BF16 autocast (FP32 на всех GPU для воспроизводимости с T4,
   возможно немного другие числа),
   включаем официальный `offload_state_to_cpu=True`. Число SAM2 instances сохраняем как диагностику.
   Без detected objects upstream может вернуть 1.0; сохраняем score, но явно предупреждаем.
   Если SAM2 CUDA extension не собрался, его официальный fallback пропускает small-hole postprocessing;
   этот запуск явно помечается и не является точной репликацией CUDA pipeline.
5. **CAS:** текущий `commonsense_adherence_score.py` вообще не читает `--finetune`.
   Создаём **официальный** `vit_giant_patch14_224` напрямую (обход `pretrained_cfg` несовместимого timm factory),
   загружаем checkpoint `module` с `strict=True`, используем официальный dataset, `final_test`, `final_merge`.
   Сохраняем 16 frames, sampling rate 4, 10 temporal segments ×3 crops; batch=1, workers=1.
   Merge делаем последовательно вместо Pool(64). DeepSpeed/training не нужны.
6. **MSS:** исходный CLI отправляет все sliding windows одним batch. Вызываем тот же scorer по одному окну
   (размер 3, включая официальный padding), затем те же threshold/artifact функции для полной последовательности.
   При 8-bit меняется точность хранения весов, что записывается в profile/source.

CSV считывает настоящие JSON keys напрямую: upstream `calculate_score.py` ошибочно ищет
`perceptible_amplitude_socre`. Для leaderboard сравнение адаптированных запусков требует отдельной валидации.


In [ ]:
runner_path = workshop.prepare_runner()
print('Отдельный процесс использует:', runner_path)


### Память, ошибки, таймер и заранее рассчитанные scores

Каждый evaluator запускается отдельным процессом. После выхода процесса все его CUDA allocations
освобождаются; `empty_cache()` в notebook не умеет освобождать память другого живого процесса.
При таймауте дочерний процесс завершается. Полный traceback находится в `metric.log`, короткое объяснение — в таблице.
Runtime inference отделён от setup/download; cache hit имеет Runtime=0, исходное время сохраняется отдельно.

Для fallback преподаватель может разместить CSV по `config.precomputed_url` или сохранить его по `workshop.precomputed_path`.
Обязательные колонки: `video_sha256,metric,score,runtime_sec,vmbench_commit,adapter_profile`.
Hash вычисляется по **конкретному файлу**; другой encode не подойдёт. Только teacher scores с той же
ревизией и profile принимаются автоматически. Никаких демонстрационных чисел не выдумываем.
Score заранее рассчитанного запуска явно маркируется `precomputed`, runtime исходного запуска не выдаётся за время Colab.


In [ ]:
workshop.load_precomputed()
print("Wrappers импортированы из utils.py; результаты хранятся в workshop.results.")


## 8. PAS — Perceptible Amplitude Score

**Насколько заметно движется главный субъект?**

```text
video → GroundingDINO (subject_noun, первый кадр) → SAM ViT-H mask
      → CoTracker3 (subject + background) → компенсация камеры
      → сумма displacement / диагональ видео → PAS (raw)
```

Upstream суммирует перемещения между декодированными кадрами, а не вычисляет скорость в m/s.
Если subject movement больше background movement, вычитает background; иначе оставляет subject movement.
Без детекции использует background motion. Это важно при real/generated и камере.
**До запуска** запишите гипотезу для original/fast/slow: почему fixed-FPS resampling может повлиять на результат?
Гарантированный ranking не задан. В этом notebook исправлен размер изображения при передаче bbox в SAM.


In [ ]:
if 'PAS' in METRICS_TO_RUN:
    pas = compute_pas(VIDEO_PATH)
    if pas is not None:
        print(f'PAS: {pas:.4f}')
else:
    print('PAS не выбран в METRICS_TO_RUN.')


**Мини-эксперимент.** Сформулируйте гипотезу, затем включите optional variants comparison в конфигурации.
Таблица показывает ту же метрику для нескольких изменённых входов. Учитывайте `source` в журнале результатов.


In [ ]:
if 'PAS' in METRICS_TO_RUN:
    compare_variants('PAS')


## 9. OIS — Object Integrity Score

**Сохраняется ли структура объекта во времени?**

```text
video → RTMDet (person, category 0) → RTMPose body7 / 17 keypoints
      → первый pose instance каждого кадра → изменения длин частей и углов суставов
      → 0.5 × length_score + 0.5 × angle_score → OIS
```

Опубликованный CLI оценивает **человеческое тело**, а не произвольные объекты. При нескольких людях
«первый instance» может соответствовать разным людям: это ограничение upstream, не identity tracker.
Низкая keypoint confidence также может давать высокий score из-за правил `pose_utils`.
Без human bbox/pose хотя бы в одном кадре адаптер возвращает N/A; missing score не равен 0.
Измените `subject_type` для животных/предметов, чтобы OIS был отмечен как неприменимый.


In [ ]:
if 'OIS' in METRICS_TO_RUN:
    ois = compute_ois(VIDEO_PATH)
    if ois is not None:
        print(f'OIS: {ois:.4f}')
else:
    print('OIS не выбран в METRICS_TO_RUN.')


**Мини-эксперимент.** Сформулируйте гипотезу, затем включите optional variants comparison в конфигурации.
Таблица показывает ту же метрику для нескольких изменённых входов. Учитывайте `source` в журнале результатов.


In [ ]:
if 'OIS' in METRICS_TO_RUN:
    compare_variants('OIS')


## 10. TCS — Temporal Coherence Score

**Не исчезают ли объекты без объяснимой причины?**

```text
video → GroundingDINO + SAM2.1 image masks → SAM2 video propagation / object IDs
      → появление / исчезновение → CoTracker3 trajectories
      → правила границы кадра, размера и detection errors → TCS
```

Код использует период detection `int(FPS)-1`; CLI `--step` затем переопределяется.
FAST/SLOW сравниваются при одинаковом FPS, чтобы не менять этот период только через metadata.
TCS держит две SAM2 модели и может хранить много masks в RAM: это самая тяжёлая tracking-секция.
Для воспроизводимости с T4 BF16 выключен на всех GPU; видео и inference state вынесены на CPU через официальные аргументы SAM2.
Если объектов не обнаружено, upstream способен вернуть 1.0 — обязательно прочитайте диагностику.


In [ ]:
if 'TCS' in METRICS_TO_RUN:
    tcs = compute_tcs(VIDEO_PATH)
    if tcs is not None:
        print(f'TCS: {tcs:.4f}')
else:
    print('TCS не выбран в METRICS_TO_RUN.')


**Мини-эксперимент.** Сформулируйте гипотезу, затем включите optional variants comparison в конфигурации.
Таблица показывает ту же метрику для нескольких изменённых входов. Учитывайте `source` в журнале результатов.


In [ ]:
if 'TCS' in METRICS_TO_RUN:
    compare_variants('TCS')


## 11. MSS — Motion Smoothness Score

**Есть ли резкие temporal artifacts?**

```text
video → окна по 3 кадра → One-Align / официальный QAlignVideoScorer
      → quality score каждого окна → абсолютные изменения соседних scores
      → threshold из PAS → доля кадров без artifacts → MSS
```

По коду используется вопрос **quality**, не отдельный aesthetic scorer. Threshold зависит от сохранённого PAS,
хотя комментарий в upstream называет его camera movement. Поэтому live MSS сначала получает PAS отдельным процессом.
На T4 исходные FP16 веса не помещаются даже при batch=1; уменьшение resolution не решает размер весов.
Official builder поддерживает `load_8bit=True`, но это меняет numerical behavior: для занятия такой вариант
нужно выбрать явно и считать экспериментальным. BitsAndBytes скачивается только в этом режиме.
Надёжный вариант воркшопа — преподаватель заранее считает MSS на GPU с достаточной памятью и раздаёт CSV.
Без этого файла показываем N/A, не фиктивный score. CPU offload через auto device map в Q-Align не проверен
для его custom multiway modules и малой RAM бесплатного Colab; здесь он не обещается как рабочая замена.

Сравните original/fast/slow/frame-drop/frame-duplication. Может ли одинаковое визуальное движение
получить разные scores после изменения количества декодированных кадров?


In [ ]:
if 'MSS' in METRICS_TO_RUN:
    mss = compute_mss(VIDEO_PATH)
    if mss is not None:
        print(f'MSS: {mss:.4f}')
else:
    print('MSS не выбран в METRICS_TO_RUN.')


**Мини-эксперимент.** Сформулируйте гипотезу, затем включите optional variants comparison в конфигурации.
Таблица показывает ту же метрику для нескольких изменённых входов. Учитывайте `source` в журнале результатов.


In [ ]:
if 'MSS' in METRICS_TO_RUN:
    compare_variants('MSS')


## 12. CAS — Commonsense Adherence Score

**Выглядит ли движение глобально правдоподобным?**

```text
video → официальный VideoMAEv2 test dataset (16 frames, 10 segments ×3 crops)
      → vit_giant_patch14_224 + vit_g_vmbench.pt
      → 5 class logits → mean softmax по test views
      → веса [0, 0.25, 0.5, 0.75, 1] → CAS
```

Это learned metric, обученная на labels качества; **CAS не является physics simulator**.
Модель не проверяет законы сохранения аналитически. Real video может оказаться вне обучающего распределения.
Checkpoint содержит ключ `module`, 40 layers, hidden size 1408 и head из 5 classes — архитектура проверена
по метаданным реального файла без скачивания всех 14 GB. Адаптер строго загружает эти веса;
мы не используем score от случайно инициализированного upstream CLI.
Сохраняем все 30 test views, не уменьшаем resolution/temporal sampling ради T4.


In [ ]:
if 'CAS' in METRICS_TO_RUN:
    cas = compute_cas(VIDEO_PATH)
    if cas is not None:
        print(f'CAS: {cas:.4f}')
else:
    print('CAS не выбран в METRICS_TO_RUN.')


**Мини-эксперимент.** Сформулируйте гипотезу, затем включите optional variants comparison в конфигурации.
Таблица показывает ту же метрику для нескольких изменённых входов. Учитывайте `source` в журнале результатов.


In [ ]:
if 'CAS' in METRICS_TO_RUN:
    compare_variants('CAS')


## 13. Контроль памяти и таблица для одного видео

В notebook остаются только числа и пути. Модели создавались в отдельных процессах и после выхода исчезли.
`peak_allocated_gib` в diagnostics — фактический PyTorch peak дочернего процесса, если inference завершился.
Он не включает всю память CUDA context и других процессов. Заранее точный peak для T4 неизвестен.

Ориентиры, **не GPU-замеры**: OIS обычно самый лёгкий; PAS — несколько GB весов плюс CoTracker activations;
TCS — примерно 2.5 GB FP32 весов плюс две SAM2 активации/маски/CoTracker; CAS — 3.77 GiB FP32 весов
плюс attention activations; MSS FP16 — ≥15.28 GiB только веса, ожидайте GPU 24 GB или больше.
Для коротких входов целевой бюджет PAS/OIS/TCS/CAS — до 15 GiB, гарантия требует Colab preflight.
MSS 8-bit может уложиться в T4, но этот inference здесь не проверен.


In [ ]:
clear_gpu(verbose=True)
workshop.show_metric_summary(VIDEO_PATH)


## 14. Все метрики последовательно — только после отдельных разделов

Теперь можно использовать convenience wrapper. Это те же пять функций по очереди.
Для повторно уже посчитанного результата работает cache. `None` означает skipped/failed/not applicable:
объяснение хранится в diagnostics. AVG не считаем: он скрывает различия между диагностическими признаками,
а опубликованный PAS в этой ревизии ещё и имеет другую шкалу.


In [ ]:
workshop.one_video_result = evaluate_video(VIDEO_PATH, METRICS_TO_RUN) if VIDEO_PATH else {'video': '(no video)'}
workshop.display(workshop.pd.DataFrame([workshop.one_video_result]))


## 15. Сравнение набора видео и CSV

Основная таблица — `video | PAS | OIS | TCS | MSS | CAS`.
`vmbench_diagnostics.csv` дополнительно сохраняет status, источник, время setup/inference, hash и peak VRAM.
Это позволяет отличить неудачный запуск от хорошего или плохого score.
Для подготовки следующего занятия `vmbench_precomputed_export.csv` содержит **только successful live** scores;
получите такой CSV на подходящем GPU и задайте его URL в начале notebook.
CSV экспортируется с N/A для missing values. Повторный Run all в той же VM переиспользует файлы и результаты,
состояние cache принадлежит объекту `workshop` и сохраняется при повторной конфигурации.


In [ ]:
results_df = workshop.evaluate_dataset()


## 16. Визуализация

Отдельные bar plots для каждой метрики. Missing values не превращаются в нули.
PAS оставляем raw; для остальных фиксируем диапазон 0–1. Не сравнивайте высоту столбцов разных метрик
как одинаковые физические величины. Radar chart не строим: normalization между этим PAS и остальными scores
скрыла бы важное отличие опубликованной реализации.


In [ ]:
workshop.plot_results(results_df)


## 17. Студенческий эксперимент

У вас есть четыре видео A, B, C, D. Среди них могут быть real, generated, fast и slow.

1. Просмотрите ролики.
2. Запишите гипотезу **до запуска** и признак, по которому вы приняли решение.
3. Посчитайте PAS/OIS/TCS/MSS/CAS (или используйте помеченные teacher scores для недоступной метрики).
4. Решите, какие метрики поддерживают вашу гипотезу.
5. Найдите хотя бы один случай, где metric кажется misleading. Проверьте applicability и detector warnings.

| Video | Human guess | PAS | OIS | TCS | MSS | CAS | Final guess |
|-------|-------------|-----|-----|-----|-----|-----|-------------|
| A | | | | | | | |
| B | | | | | | | |
| C | | | | | | | |
| D | | | | | | | |

Эти метрики оценивают свойства движения, а не обученный real/generated classifier.
Допускается «не знаю» в учебной таблице; не превращайте N/A в плохой score.


In [ ]:
student_df = workshop.student_table()


## 18. Optional: шаблон leaderboard submission

Это файл для учебного классификационного задания, **не формат официального VMBench leaderboard**.
Сервера отправки нет. Допустимые labels: REAL_NORMAL, GENERATED, FAST, SLOW.
Confidence — ваше число от 0 до 1, а не калиброванная вероятность из VMBench.
Пустой template намеренно невалиден до заполнения, validator объяснит причину без длинного traceback.


In [ ]:
submission_df = workshop.submission_template()
# После заполнения: validate_submission(submission_df)


## 19. Обсуждение

1. Какая metric лучше всего обнаружила изменение speed?
2. Какая metric почти не изменилась?
3. Можно ли по PAS определить real/generated?
4. Может ли synthetic video получить высокий MSS?
5. Может ли real video получить плохой CAS?
6. Почему одна aggregate metric может быть опасной?
7. Где metric disagreed with your visual judgement?
8. Чем отличаются отсутствие субъекта, низкий score и ошибка evaluator’а?
9. Какие различия могли быть вызваны preprocessing, codec или 8-bit precision?

После Runtime reset `/content` исчезает: повторите Run all, снова загрузите видео или используйте demo URLs.
Скачайте CSV заранее. Все зависимости/checkpoint’ы восстанавливаются автоматически для запускаемых метрик.
Для преподавателя: подробности, известные ограничения и проверенные операции — в соседнем `README.md`.
